In [1]:
%pip install -U langgraph langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import re
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, START, END
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
from getpass import getpass
# Carregar variáveis de ambiente
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "OPENAI_API_KEY: "
    ).strip()
# Inicializar o modelo de linguagem
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# Definir a estrutura de dados do candidato
class Candidate(TypedDict):
    name: str
    resume: str
    skills: List[str]
    experience: int
    education_level: str
    score: float
    feedback: str

# Definir o estado do fluxo de trabalho
class State(TypedDict):
    candidates: List[Candidate]
    job_requirements: Dict[str, any]
    result: str



In [7]:

import re

def extract_information_from_resume(resume_text: str) -> dict:
    """Extrai informações do currículo usando regex."""
    info = {}

    # Extrair nome
    name_match = re.search(
        r'Nome:\s*([^\n]+)', resume_text, re.IGNORECASE
    )
    info['name'] = name_match.group(1).strip() if name_match else ''

    # Extrair experiência
    exp_match = re.search(
        r'(\d+)\s*anos? de experiência',
        resume_text, re.IGNORECASE
    )
    info['experience'] = int(exp_match.group(1)) if exp_match else 0

    # Extrair educação
    edu_patterns = [
        r'\bp[oó]s[-\s]*doutorand[oa]\b(?:\s+em\s+[^.,\n]+)?',
        r'\bp[oó]s[-\s]*doutorado\b(?=[^.\n]*\bem andamento\b)[^.\n]*',
        r'\bp[oó]s[-\s]*doutorado\b(?:\s+em\s+[^.,\n]+)?',
        r'\bdoutorand[oa]\b(?:\s+em\s+[^.,\n]+)?',
        r'\bdoutorado\b(?=[^.\n]*\bem andamento\b)[^.\n]*',
        r'\bdoutorado em [^.,\n]+',
        r'\bdoutor(?:a)? em [^.,\n]+',
        r'\bmestrand[oa]\b(?:\s+em\s+[^.,\n]+)?',
        r'\bmestrado\b(?=[^.\n]*\bem andamento\b)[^.\n]*',
        r'\bmestrado em [^.,\n]+',
        r'\bmestre em [^.,\n]+',
        r'\bgraduação em [^.,\n]+',
        r'\b(?:rec[eé]m[-\s]?)?formad[ao] em [^.,\n]+'
    ]

    info['education_level'] = ''

    for pattern in edu_patterns:
        edu_match = re.search(
            pattern, resume_text, re.IGNORECASE
        )

        if edu_match:
            info['education_level'] = edu_match.group(0).strip()
            break

    # Extrair habilidades
    skills = []

    skill_patterns = [
        r'proficiente em ([^.\n]+)',
        r'conhecimento em ([^.\n]+)',
        r'especialista em ([^.\n]+)',
        r'experiência (?:com|em) ([^.\n]+)'
    ]

    for pattern in skill_patterns:
        skill_match = re.search(
            pattern, resume_text, re.IGNORECASE
        )

        if skill_match:
            texto = skill_match.group(1)

            skills.extend(
                s.strip()
                for s in re.split(r',|\s+e\s+', texto)
                if s.strip()
            )

    info['skills'] = list(dict.fromkeys(skills))

    return info


In [15]:

import re

def score_education(education_level: str) -> float:
    """Calcula a pontuação acadêmica do candidato."""

    edu_patterns = [
        r'\bp[oó]s[-\s]*doutorand[oa]\b',
        r'\bp[oó]s[-\s]*doutorado\b(?=[^.,\n]*\bem andamento\b)',
        r'\bp[oó]s[-\s]*doutorado\b(?=[^.,\n]*\bconclu[ií]d[oa]\b)',
        r'\bp[oó]s[-\s]*doutorado\b',
        r'\bdoutorand[oa]\b',
        r'\bdoutorado\b(?=[^.,\n]*\bem andamento\b)',
        r'\bdoutorado\b',
        r'\bdoutor(?:a)?\b',
        r'\bmestrand[oa]\b',
        r'\bmestrado\b(?=[^.,\n]*\bem andamento\b)',
        r'\bmestrado\b',
        r'\bmestre\b',
        r'\bgraduação\b',
        r'\b(?:rec[eé]m[-\s]?)?formad[ao] em\b'
    ]

    # Pontuação correspondente a cada padrão
    edu_scores = [
        [12.5],                  # Pós-doutorando
        [12.5],                  # Pós-doutorado em andamento
        [15.0],                  # Pós-doutorado concluído
        [15.0],                  # Pós-doutorado
        [7.5],                   # Doutorando
        [7.5],                   # Doutorado em andamento
        [10.0],                  # Doutorado concluído
        [10.0],                  # Doutor
        [4.5],                   # Mestrando
        [4.5],                   # Mestrado em andamento
        [6.0],                   # Mestrado concluído
        [6.0],                   # Mestre
        [3.0],                   # Graduação
        [3.0]                    # Formado
    ]

    # Verificar formação e somar as pontuações
    for pattern, scores in zip(edu_patterns, edu_scores):
        edu_match = re.search(
            pattern, education_level or "", re.IGNORECASE
        )

        if edu_match:
            return sum(scores)

    return 0.0


In [16]:
for candidate in candidates:
    info = extract_information_from_resume(candidate['resume'])

    print("Nome:", info['name'])
    print("Formação:", info['education_level'])
    print("Nota acadêmica:", score_education(info['education_level']))
    print("Experiência:", info['experience'])
    print("-" * 40)

Nome: Eugenio JR
Formação: pós doutorando em engenharia de teleinformática
Nota acadêmica: 12.5
Experiência: 10
----------------------------------------
Nome: João Souza
Formação: mestrado em Engenharia de Software
Nota acadêmica: 6.0
Experiência: 8
----------------------------------------
Nome: Ana Pereira
Formação: recém-formada em Sistemas de Informação
Nota acadêmica: 3.0
Experiência: 0
----------------------------------------


In [17]:
def evaluate_candidate(state: State) -> State:
    """Avalia os candidatos com base nos requisitos."""
    updated_candidates = []
    
    for candidate in state['candidates']:
        try:
            # Sistema de pontuação direto
            score = 0
            feedback_points = []
            
            # Avaliar habilidades (4 pontos)
            required_skills = state['job_requirements']['skills_required']
            matching_skills = []
            for req_skill in required_skills:
                for candidate_skill in candidate['skills']:
                    if req_skill.lower() in candidate_skill.lower():
                        matching_skills.append(req_skill)
                        break
            
            skill_score = len(set(matching_skills)) / len(required_skills) * 4
            score += skill_score
            feedback_points.append(f"Possui {len(set(matching_skills))} de {len(required_skills)} habilidades requeridas")
            
            # Avaliar experiência (3 pontos)
            min_exp = state['job_requirements']['minimum_experience']
            if candidate['experience'] >= min_exp:
                score += 3
                feedback_points.append(f"Experiência adequada ({candidate['experience']} anos)")
            else:
                feedback_points.append(f"Experiência abaixo do requisito ({candidate['experience']} anos)")
            
            # Avaliar educação (3 pontos)
            if candidate['education_level']:
                score += 3
                feedback_points.append(f"Possui formação adequada: {candidate['education_level']}")
            
            # Atualizar candidato
            candidate['score'] = min(score, 10)  # Limita a 10 pontos
            candidate['feedback'] = ". ".join(feedback_points)
            
        except Exception as e:
            candidate['score'] = 0.0
            candidate['feedback'] = f"Erro na avaliação: {str(e)}"
        
        updated_candidates.append(candidate)
    
    state['candidates'] = updated_candidates
    return state

def rank_candidates(state: State) -> State:
    """Classifica os candidatos por pontuação."""
    try:
        state['candidates'].sort(key=lambda x: (float(x.get('score', 0)), x.get('experience', 0)), reverse=True)
    except Exception as e:
        print(f"Erro ao classificar candidatos: {e}")
    return state

In [18]:

def generate_result(state: State) -> State:
    """Gera relatório com nota acadêmica e pontuação final."""
    try:
        if not state['candidates']:
            state['result'] = 'Não há candidatos para avaliar.'
            return state

        report = []
        report.append("=== RELATÓRIO DE CLASSIFICAÇÃO DE CANDIDATOS ===\n")

        # Requisitos da vaga
        req = state['job_requirements']
        report.append("REQUISITOS DA VAGA:")
        report.append(f"- Habilidades: {', '.join(req['skills_required'])}")
        report.append(f"- Experiência mínima: {req['minimum_experience']} anos")
        report.append(f"- Educação: {req['education_level']}")
        report.append(f"- Atributos desejados: {', '.join(req['desired_attributes'])}\n")

        # Melhor candidato
        best = state['candidates'][0]
        nota_academica = score_education(best.get('education_level', ''))

        report.append("MELHOR CANDIDATO:")
        report.append(f"Nome: {best.get('name', 'Não disponível')}")
        report.append(f"Nota acadêmica: {nota_academica:.1f}/10")
        report.append(f"Pontuação final: {best.get('score', 0):.1f}/10")
        report.append(f"Experiência: {best.get('experience', 0)} anos")
        report.append(f"Educação: {best.get('education_level', 'Não informado')}")
        report.append(f"Habilidades: {', '.join(best.get('skills', []))}")
        report.append(f"Feedback: {best.get('feedback', 'Sem feedback')}\n")

        # Ranking completo
        report.append("=== RANKING COMPLETO ===\n")

        for i, candidate in enumerate(state['candidates'], 1):
            nota_academica = score_education(
                candidate.get('education_level', '')
            )
            nota_final = candidate.get('score', 0)

            report.append(f"{i}. {candidate.get('name', f'Candidato {i}')}")
            report.append(f"   Formação: {candidate.get('education_level', 'Não informada')}")
            report.append(f"   Nota acadêmica: {nota_academica:.1f}/10")
            report.append(f"   Pontuação final: {nota_final:.1f}/10")
            report.append("")

        state['result'] = '\n'.join(report)

    except Exception as e:
        state['result'] = f"Erro ao gerar relatório: {str(e)}"

    return state
def extract_information(state: State) -> dict:
    """Extrai informações de todos os candidatos."""
    updated_candidates = []

    for candidate in state["candidates"]:
        info = extract_information_from_resume(
            candidate["resume"]
        )

        if not isinstance(info, dict):
            raise TypeError(
                "extract_information_from_resume deve retornar um dicionário"
            )

        updated_candidates.append({
            **candidate,
            **info,
            "score": 0.0,
            "feedback": ""
        })

    return {"candidates": updated_candidates}

# Construir o workflow
workflow = StateGraph(State)

# Adicionar nós
workflow.add_node("extract_information", extract_information)
workflow.add_node("evaluate_candidate", evaluate_candidate)
workflow.add_node("rank_candidates", rank_candidates)
workflow.add_node("generate_result", generate_result)

# Definir as arestas
workflow.add_edge(START, "extract_information")
workflow.add_edge("extract_information", "evaluate_candidate")
workflow.add_edge("evaluate_candidate", "rank_candidates")
workflow.add_edge("rank_candidates", "generate_result")
workflow.add_edge("generate_result", END)

# Compilar o grafo
app = workflow.compile()

def process_candidates(candidates: List[Candidate], job_requirements: Dict[str, any]) -> str:
    """Processa os candidatos e retorna o resultado."""
    try:
        initial_state = State(
            candidates=candidates,
            job_requirements=job_requirements,
            result=""
        )
        
        final_state = app.invoke(initial_state)
        return final_state.get('result', 'Erro: Resultado não disponível')
    except Exception as e:
        return f"Erro durante o processamento: {str(e)}"

# Exemplo de uso
if __name__ == "__main__":
    # Exemplo de candidatos
    candidates = [
        {
            "resume": """
            Nome: Eugenio JR
            
            Eugenio JR está estudando consistentemente o curso compreto da Scoras Academy para se expecializar em IA. 
            El é proficiente em Python, Entende profundamente de LangGrapg, PydanticAI, SLMs e LLMs e tem experiência com bancos de dados SQL. 
            Ele é engenheiro mecatrônico, mestre em informática aplicada edoutor em engenharia elétrica, pós doutorando em engenharia de teleinformática.
            Está com 10 anos de experiência em desenvolvimento de software.
            """
        },
        {
            "resume": """
            Nome: João Souza
            
            João é engenheiro de software com 8 anos de experiência. 
            Especialista em Java e C++. 
            Possui mestrado em Engenharia de Software.
            Tem experiência em liderança de equipes e desenvolvimento de sistemas de grande escala.
            """
        },
        {
            "resume": """
            Nome: Ana Pereira
            
            Ana é recém-formada em Sistemas de Informação. 
            Possui conhecimento em Python, HTML e CSS. 
            Participou de projetos acadêmicos relacionados a desenvolvimento web.
            Trabalhou em projetos de desenvolvimento web e está familiarizada com metodologias ágeis.
            """
        }
    ]

    # Requisitos da vaga
    job_requirements = {
        "skills_required": ["Python", "SQL", "Desenvolvimento Web", "LangGraph", "PydanticAI", "LLMs", "SLMs"],
        "minimum_experience": 3,
        "education_level": "Graduação em área relacionada",
        "desired_attributes": ["Trabalho em equipe", "Metodologias Ágeis"]
    }

    # Processar os candidatos
    result = process_candidates(candidates, job_requirements)
    print(result)

=== RELATÓRIO DE CLASSIFICAÇÃO DE CANDIDATOS ===

REQUISITOS DA VAGA:
- Habilidades: Python, SQL, Desenvolvimento Web, LangGraph, PydanticAI, LLMs, SLMs
- Experiência mínima: 3 anos
- Educação: Graduação em área relacionada
- Atributos desejados: Trabalho em equipe, Metodologias Ágeis

MELHOR CANDIDATO:
Nome: Eugenio JR
Nota acadêmica: 12.5/10
Pontuação final: 8.9/10
Experiência: 10 anos
Educação: pós doutorando em engenharia de teleinformática
Habilidades: Python, Entende profundamente de LangGrapg, PydanticAI, SLMs, LLMs, tem experiência com bancos de dados SQL, bancos de dados SQL
Feedback: Possui 5 de 7 habilidades requeridas. Experiência adequada (10 anos). Possui formação adequada: pós doutorando em engenharia de teleinformática

=== RANKING COMPLETO ===

1. Eugenio JR
   Formação: pós doutorando em engenharia de teleinformática
   Nota acadêmica: 12.5/10
   Pontuação final: 8.9/10

2. João Souza
   Formação: mestrado em Engenharia de Software
   Nota acadêmica: 6.0/10
   Pontuaç